# Recomendador basado en contenido — Estrategia 2: LLM local (Ollama)

**Introducción al Procesamiento del Lenguaje Natural — TFI 1C 2026**

Esta es la **segunda estrategia de representación**, para comparar contra el TF-IDF de `recomendador_tfidf.ipynb`. En lugar de representar películas y usuarios como vectores léxicos, delegamos el "entendimiento" a un **LLM local** (vía **Ollama**) que razona sobre el *significado* de la query, el historial y las sinopsis candidatas.

**La diferencia de fondo con TF-IDF:** TF-IDF mide coincidencia de palabras; el LLM opera sobre el sentido, así que puede conectar *"una película que te haga pensar sobre qué es real"* con sinopsis que nunca usan esas palabras. El precio: es más lento, menos transparente y no determinista.

### Arquitectura: recuperación + reordenamiento (RAG)

Un LLM no puede leer las ~5.000 sinopsis en su ventana de contexto. Por eso usamos un esquema en dos pasos:

1. **Recuperación (TF-IDF):** preseleccionamos un *shortlist* de candidatos plausibles. Barato y rápido.
2. **Reordenamiento (LLM):** el modelo lee query + historial + las sinopsis del shortlist y **elige las 5 mejores con una justificación**.

Así el LLM aporta donde es fuerte (juicio semántico) sobre un conjunto manejable.


## 0. Requisitos — Ollama

Este notebook necesita **Ollama corriendo localmente**. En tu máquina:

```bash
# 1. Instalar Ollama (https://ollama.com/download) y luego descargar un modelo:
ollama pull llama3.1        # ~4.7 GB ; alternativas: qwen2.5, mistral, gemma2
# 2. Ollama levanta un servidor en http://localhost:11434 automáticamente.
```

No hace falta API key: corre todo en tu equipo. Las celdas detectan si Ollama está disponible; si no lo está, igual muestran el *prompt* que se enviaría, para que se vea el diseño.

## 1. Datos y verificación de Ollama

In [14]:
import os, re, json
import numpy as np
import pandas as pd
import requests

DATA = "data"
MODEL = "llama3.1"                  # cambiá por el modelo que tengas en `ollama list`
OLLAMA_URL = "http://localhost:11434"

df = pd.read_csv(os.path.join(DATA, "plots.csv")).reset_index(drop=True)
users = pd.read_csv(os.path.join(DATA, "perfiles_usuarios.csv"))
mapa = pd.read_csv(os.path.join(DATA, "mapa_titulos.csv"), index_col=0)["titulo_corpus"].to_dict()

def ollama_disponible():
    try:
        r = requests.get(f"{OLLAMA_URL}/api/tags", timeout=2)
        return r.status_code == 200
    except Exception:
        return False

OLLAMA_OK = ollama_disponible()
print("Ollama disponible:", OLLAMA_OK)
if OLLAMA_OK:
    modelos = [m["name"] for m in requests.get(f"{OLLAMA_URL}/api/tags").json().get("models", [])]
    print("Modelos instalados:", modelos)
else:
    print("→ Ollama no detectado. Las celdas de recuperación corren igual;")
    print("  las de generación mostrarán el prompt en lugar de llamar al modelo.")

Ollama disponible: True
Modelos instalados: ['llama3.1:latest']


## 2. Recuperación: shortlist con TF-IDF

Reutilizamos la representación léxica para preseleccionar candidatos. Es el mismo preprocesamiento del notebook anterior (stopwords de la materia + inglés, stemming), sobre `description + keywords + genre`.

In [15]:
from unidecode import unidecode
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

sw_es = set(pd.read_csv(os.path.join(DATA, "stop_words_spanish.csv"))["word"].astype(str))
STOPWORDS = {unidecode(w).lower() for w in sw_es} | set(ENGLISH_STOP_WORDS)
stemmer = SnowballStemmer("spanish")

def preprocesar(texto):
    t = unidecode(str(texto)).lower()
    t = re.sub(r"[^a-z0-9\s]", " ", t)
    t = re.sub(r"\w*\d\w*", " ", t)
    toks = [stemmer.stem(w) for w in t.split() if w not in STOPWORDS and len(w) > 2]
    return " ".join(toks)

df["doc"] = (df["description"].map(preprocesar) + " " +
             (df["keywords"].map(preprocesar) + " ") * 2 +
             (df["genre"].map(preprocesar) + " ") * 2)

vectorizer = TfidfVectorizer(min_df=2, max_df=0.6, ngram_range=(1, 2), sublinear_tf=True)
M = vectorizer.fit_transform(df["doc"])
name_to_idx = {n: i for i, n in enumerate(df["name"])}
hist_cols = [c for c in users.columns if c.startswith("pelicula_")]
print("Matriz TF-IDF lista:", M.shape)

Matriz TF-IDF lista: (4967, 15520)


In [16]:
def indices_historial(row):
    out = []
    for c in hist_cols:
        t = row[c]
        if pd.notna(t) and mapa.get(t) in name_to_idx:
            out.append(name_to_idx[mapa[t]])
    return out

def shortlist(row, k=25):
    idxs = indices_historial(row)
    v_hist = normalize(np.asarray(M[idxs].mean(axis=0)).reshape(1, -1)) if idxs else None
    vq = vectorizer.transform([preprocesar(row["query"])])
    v_query = normalize(vq).toarray() if vq.nnz > 0 else None
    if v_hist is not None and v_query is not None:
        v = 0.5 * v_hist + 0.5 * v_query
    else:
        v = v_hist if v_hist is not None else v_query
    sims = cosine_similarity(v, M).flatten()
    excluir = set(idxs)
    cands = [i for i in np.argsort(-sims) if i not in excluir][:k]
    return cands

# Verificación: shortlist para un usuario
cands = shortlist(users.iloc[1])
print("Shortlist de Rodrigo (primeros 8):")
for i in cands[:8]:
    print(" •", df.iloc[i]["name"], "—", df.iloc[i]["genre"])

Shortlist de Rodrigo (primeros 8):
 • Gangs of New York — crimen, drama
 • Buenas noches, y buena suerte. — biografía, drama, historia
 • 2013: Rescate en L.A. — acción, aventura, ciencia ficción
 • El expreso de Elmira — biografía, drama, deporte
 • La cortina de humo — comedia, drama
 • Maleficio — drama, historia, terror
 • El asesinato de Richard Nixon — biografía, crimen, drama
 • La verdad oculta — biografía, crimen, drama


## 3. Reordenamiento con el LLM

Construimos un prompt que le da al modelo el rol, el contexto del usuario (historial + query) y el shortlist con sinopsis, y le pedimos que devuelva **JSON** con las 5 elegidas y una justificación corta. Pedir JSON (`format=json`) hace la salida parseable.

In [17]:
def construir_prompt(row, cands):
    historial = []
    for c in hist_cols:
        t = row[c]
        if pd.notna(t) and mapa.get(t) in name_to_idx:
            historial.append(df.iloc[name_to_idx[mapa[t]]]["name"])
    candidatos = []
    for i in cands:
        r = df.iloc[i]
        sinopsis = str(r["description"])[:180]
        candidatos.append(f'- {r["name"]} [{r["genre"]}]: {sinopsis}')
    candidatos_txt = "\n".join(candidatos)

    system = (
        "Sos un sistema de recomendación de películas. A partir del historial de un "
        "usuario y de lo que pide en lenguaje natural, elegís las 5 películas más "
        "adecuadas SOLO de la lista de candidatos. Priorizá la intención de la query; "
        "usá el historial para entender el gusto. Respondé en JSON."
    )
    user = f"""HISTORIAL (lo que ya vio): {", ".join(historial)}

LO QUE QUIERE VER AHORA: "{row['query']}"

CANDIDATOS (elegí 5 de acá, por nombre exacto):
{candidatos_txt}

Devolvé un JSON con esta forma exacta:
{{"recomendaciones": [{{"pelicula": "<nombre exacto>", "motivo": "<una frase>"}}]}}"""
    return system, user

# Mostramos el prompt para Rodrigo (sirva o no Ollama)
s, u = construir_prompt(users.iloc[1], shortlist(users.iloc[1]))
print(u[:900], "\n...")

HISTORIAL (lo que ya vio): Adiós Bafana, Mi pie izquierdo, L.A. Confidential, Érase una vez en América, El juego del halcón

LO QUE QUIERE VER AHORA: "Busco algo basado en hechos reales sobre corrupción o poder político"

CANDIDATOS (elegí 5 de acá, por nombre exacto):
- Gangs of New York [crimen, drama]: En 1862, Amsterdam Vallon regresa a la zona de Five Points, en Nueva York, en busca de venganza contra Bill el Carnicero, el asesino de su padre.
- Buenas noches, y buena suerte. [biografía, drama, historia]: &quot;Basada en hechos reales, narra la historia del famoso periodista de la CBS Edward R. Murrow (David Strathairn) y su productor Fred Friendly (George Clooney) durante la legend
- 2013: Rescate en L.A. [acción, aventura, ciencia ficción]: El gobierno de los Estados Unidos recurre de nuevo a Snake Plissken para recuperar un dispositivo que podría acabar con el mundo, en un futuro 
...


In [18]:
def recomendar_llm(row, k=25):
    cands = shortlist(row, k=k)
    system, user = construir_prompt(row, cands)
    if not OLLAMA_OK:
        return {"_status": "ollama_no_disponible", "prompt_preview": user[:300]}
    payload = {
        "model": MODEL,
        "messages": [{"role": "system", "content": system},
                     {"role": "user", "content": user}],
        "format": "json",
        "stream": False,
        "options": {"temperature": 0.2},
    }
    r = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=120)
    contenido = r.json()["message"]["content"]
    try:
        return json.loads(contenido)
    except json.JSONDecodeError:
        return {"_status": "json_invalido", "raw": contenido}

## 4. Recomendaciones para los usuarios

Si Ollama está corriendo, esto genera las recomendaciones. Mostramos dos perfiles de contraste (uno definido, uno ambiguo) y, si querés, el loop completo sobre los 14.

In [19]:
for nombre in ["Rodrigo", "Paula"]:
    row = users[users["nombre"] == nombre].iloc[0]
    print("="*70)
    print(f"{row['nombre']} ({row['tipo_perfil']}) — \"{row['query']}\"")
    res = recomendar_llm(row)
    if res.get("_status") == "ollama_no_disponible":
        print("  [Ollama no disponible en esta ejecución — correr localmente]")
        print("  Prompt (preview):", res["prompt_preview"][:160], "...")
    else:
        for rec in res.get("recomendaciones", []):
            print(f"   • {rec.get('pelicula','?')} — {rec.get('motivo','')}")

Rodrigo (definido) — "Busco algo basado en hechos reales sobre corrupción o poder político"
   • Buenas noches, y buena suerte. — Basada en hechos reales sobre corrupción política.
   • El asesinato de Richard Nixon — Basado en hechos de la vida real sobre el poder político.
   • La verdad oculta — Basada en las experiencias de Kathryn Bolkovac, una policía que denunció públicamente a Naciones Unidas por encubrir corrupción.
   • El lobo de Wall Street — Basado en la historia real de Jordan Belfort y su caída debido al crimen y la corrupción.
   • Lincoln — Basada en hechos reales sobre el presidente de Estados Unidos que lucha con la corrupción política durante la Guerra Civil Americana.
Paula (ambiguo) — "No tengo ganas de pensar mucho, pero tampoco quiero algo vacío, algo intermedio"
   • Beautiful Girls — La comedia y el drama en esta película pueden equilibrar la falta de profundidad que mencionas
   • El último unicornio — La aventura y la fantasía en esta película pueden ser lo 

In [ ]:
# Loop completo sobre los 14 usuarios (descomentar para correr con Ollama)
resultados = {}
for _, row in users.iterrows():
    resultados[row["nombre"]] = recomendar_llm(row)
    print(row["nombre"], "listo")
print("Loop completo disponible arriba (comentado) para ejecución local con Ollama.")

Valentina listo
Rodrigo listo
Camila listo
Tomás listo
Lucía listo
Martín listo
Sofía listo
Diego listo
Elena listo
Facundo listo
Julián listo
Mariana listo
Nicolás listo
Paula listo
{
  "Valentina": {
    "recomendaciones": [
      {
        "pelicula": "El ente",
        "motivo": "Una mujer es atormentada y abusada sexualmente por un demonio invisible, lo que se ajusta a la amenaza invisible de alguien cercano mencionada en la query"
      },
      {
        "pelicula": "Misteriosa obsesión",
        "motivo": "Después de que les dicen que sus hijos nunca existieron, un hombre y una mujer pronto descubren que hay un enemigo mucho más grande en el trabajo, lo que sugiere una amenaza invisible"
      },
      {
        "pelicula": "El silencio de los corderos",
        "motivo": "Una joven cadete del FBI busca la ayuda de un asesino caníbal y manipulador encarcelado con el fin de atrapar a otro asesino en serie, lo que implica una amenaza invisible"
      },
      {
        "pelicula"

In [23]:
import json

print(json.dumps(resultados, ensure_ascii=False, indent=2))

{
  "Valentina": {
    "recomendaciones": [
      {
        "pelicula": "El ente",
        "motivo": "Una mujer es atormentada y abusada sexualmente por un demonio invisible, lo que se ajusta a la amenaza invisible de alguien cercano mencionada en la query"
      },
      {
        "pelicula": "Misteriosa obsesión",
        "motivo": "Después de que les dicen que sus hijos nunca existieron, un hombre y una mujer pronto descubren que hay un enemigo mucho más grande en el trabajo, lo que sugiere una amenaza invisible"
      },
      {
        "pelicula": "El silencio de los corderos",
        "motivo": "Una joven cadete del FBI busca la ayuda de un asesino caníbal y manipulador encarcelado con el fin de atrapar a otro asesino en serie, lo que implica una amenaza invisible"
      },
      {
        "pelicula": "Aflicción",
        "motivo": "Un policía de un pequeño pueblo investiga una muerte sospechosa de caza mientras otros eventos ponen en peligro su cordura, lo que sugiere una amenaz

## 5. Comparación con TF-IDF (lo que va al informe)

| Aspecto | TF-IDF (Estrategia 1) | LLM local (Estrategia 2) |
|---|---|---|
| Qué compara | Coincidencia de palabras (léxico) | Significado / intención |
| Negación y matices ("no muy larga") | No los entiende | Los puede interpretar |
| Queries abstractas ("hacer pensar sobre qué es real") | Falla si no comparte vocabulario | Puede conectar el concepto |
| Transparencia | Alta (se ven los términos) | Baja (caja negra) + da un *motivo* |
| Determinismo / reproducibilidad | Total | No determinista (temperatura, modelo) |
| Costo | Milisegundos | Segundos por usuario |
| Riesgo propio | Recomendaciones literales | *Alucinar* títulos fuera de la lista |

**Por qué el LLM reordena en vez de recomendar desde cero:** si lo dejáramos elegir entre las 5.000 películas de memoria, inventaría títulos o recomendaría películas que no están en el corpus. Restringirlo al shortlist (recuperado por TF-IDF) lo ancla a datos reales. Es decir: **las dos estrategias no compiten, se complementan** — pero para el informe las evaluamos como representaciones distintas y discutimos dónde cada una acierta y falla sobre los mismos usuarios.

**Qué mirar al comparar resultados:** tomá un perfil definido (p. ej. Rodrigo) y uno ambiguo (p. ej. Paula) y compará las 5 recomendaciones de cada estrategia. La hipótesis es que en los definidos ambas coinciden bastante, y en los ambiguos el LLM aprovecha mejor la intención difusa de la query mientras TF-IDF se queda en lo léxico.
